# AksharaPiece on Kaggle: what the tokenizer changes in practice

Bits per byte showed no reliable difference between plain BPE and AksharaPiece. This notebook tests the two things a user of the model would notice, using the **8 models already trained** in the seeds notebook (no new pretraining):

1. **Broken syllables in generated text.** Each model continues 300 Nepali sentence openings. We count malformed syllables (a vowel sign or halant with nothing to attach to, or two vowel signs on one syllable) and words that never occur in real Nepali text. Human-written text is scored the same way.
2. **News classification.** Each model is fine-tuned on the Nepali news dataset np20ng (20 topics, ~1,000 articles per topic) and scored on accuracy and macro-F1.

**Before you save and run**

1. **Create a new notebook** and upload this file (File → Import Notebook).
2. **Attach the trained models:** right-hand panel → **Add Input** → **Notebooks** (or "Your Work") → pick **aksharapiece_seeds** (the notebook whose output has `models/` and `aksharapiece_lm_results.zip`) → **Add**.
3. **GPU T4 x2** under Session options → Accelerator, and **Internet → On**.
4. **Hugging Face:** Add-ons → Secrets → switch on `HF_TOKEN` (your Read token) for this notebook. Then, logged in on huggingface.co, open **huggingface.co/datasets/Suyogyart/np20ng** and click **Agree / Access repository** once (approval is automatic). Without this, part 2 is skipped and part 1 still runs.
5. Run the first three code cells by hand: the setup cell must list two T4 GPUs and **8 models**. Then **Save Version → Save & Run All (Commit)**, and stop the editor session.

**Time:** about 30–60 minutes (both parts run at once, one per GPU), so about 1–2 hours of GPU quota.

In [ ]:
# ---- Settings ----
N_PROMPTS = 300          # sentence openings each model continues
PROMPT_WORDS = 6         # words of real text given as the opening
MAX_NEW = 64             # tokens generated per opening
WORDLIST_MB = 200        # real Nepali text used to decide what counts as a real word
CLS_TRAIN_PER_CLASS = 1000
CLS_TEST_PER_CLASS = 200
CLS_EPOCHS = 3
ALLOW_CPU = False        # keep False: without a GPU the notebook stops

## Setup
Unpacks the project code, checks the GPUs and finds the trained models from the seeds notebook.

In [ ]:
import base64, glob, io, json, os, shutil, subprocess, sys, time, zipfile

ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = "/kaggle/working" if ON_KAGGLE else os.getcwd()
WORK = "/tmp/aksharapiece_eval" if ON_KAGGLE else os.path.join(os.getcwd(), "work")
INPUT = "/kaggle/input" if ON_KAGGLE else os.environ.get("EVAL_INPUT", os.getcwd())
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
CODE_ZIP_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP_B64))).extractall(WORK)
PROJECT = os.path.join(WORK, "aksharapiece")
os.chdir(PROJECT)
sys.path[:0] = [PROJECT, os.path.join(PROJECT, "scripts")]
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = PROJECT


def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")


import torch
N_GPU = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "none")
if N_GPU == 0 and not ALLOW_CPU:
    raise RuntimeError("No GPU found. Open Session options -> Accelerator -> GPU T4 x2, then run again.")

# trained models + tokenizers from the seeds notebook (attached with Add Input)
MODELS = sorted(glob.glob(os.path.join(INPUT, "**", "models", "*.pt"), recursive=True))
zips = glob.glob(os.path.join(INPUT, "**", "aksharapiece_lm_results.zip"), recursive=True)
if not MODELS or not zips:
    raise RuntimeError("Trained models not found. Add Input -> Notebooks -> aksharapiece_seeds (its output must "
                       "contain models/ and aksharapiece_lm_results.zip), then run again.")
MODEL_DIR = os.path.dirname(MODELS[0])
TOK_DIR = os.path.join(WORK, "tokenizers")
with zipfile.ZipFile(zips[0]) as z:
    for n in z.namelist():
        if n.startswith("tokenizers/") and not n.endswith("/"):
            z.extract(n, WORK)
print(f"{len(MODELS)} models in {MODEL_DIR}:")
for m in MODELS:
    print("  ", os.path.basename(m), f"{os.path.getsize(m) / 1e6:.0f} MB")
print("tokenizers:", sorted(os.listdir(TOK_DIR)))

In [ ]:
run("python -m pytest -q tests")

In [ ]:
HF_OK = False
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=_t)
    os.environ["HF_TOKEN"] = _t        # for the classification script (never printed)
    del _t
    HF_OK = True
    print("Logged in to Hugging Face.")
except Exception:
    print("No HF_TOKEN secret: part 2 (classification) will most likely be skipped.")

## Step 1. Real Nepali text for comparison
Downloads a slice of IndicCorp v2 from **far beyond** the part used for training (which was the first 1.5 GB). One part becomes the list of real words, a separate part is the human-written reference that gets the same scores as the models.

In [ ]:
import re
import requests
from aksharapiece.akshara import normalize


def download_range(url, dest, start, length, token=None, attempts=8):
    """Download bytes [start, start+length) with resume; keep only complete lines."""
    got = os.path.getsize(dest) if os.path.exists(dest) else 0
    t0 = time.time()
    for attempt in range(1, attempts + 1):
        if got >= length:
            break
        h = {"Range": f"bytes={start + got}-{start + length - 1}"}
        if token:
            h["Authorization"] = f"Bearer {token}"
        try:
            with requests.get(url, headers=h, stream=True, timeout=(30, 120)) as r:
                if r.status_code != 206:
                    raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
                with open(dest, "ab") as f:
                    for chunk in r.iter_content(8 << 20):
                        f.write(chunk)
                        got += len(chunk)
        except Exception as e:
            print(f"  attempt {attempt}: {type(e).__name__}: {str(e)[:150]} - resuming in 10 s", flush=True)
            time.sleep(10)
    if got < length * 0.99:
        raise RuntimeError(f"download incomplete: {got / 1e6:.0f} of {length / 1e6:.0f} MB")
    print(f"  {got / 1e6:,.0f} MB in {time.time() - t0:.0f}s", flush=True)


DEVA = re.compile(r"[\u0900-\u097F]")


def clean_lines(raw, dest, max_lines=None):
    n = 0
    with open(raw, encoding="utf-8", errors="ignore") as f, open(dest, "w", encoding="utf-8") as out:
        next(f, None)                                  # first line may be cut in the middle
        for line in f:
            t = normalize(line)
            if len(t) >= 20 and len(DEVA.findall(t)) >= 0.6 * len(t.replace(" ", "")):
                out.write(t + "\n")
                n += 1
                if max_lines and n >= max_lines:
                    break
    return n


os.makedirs("data/ref", exist_ok=True)
if not os.path.exists("data/ref/words.txt"):
    from huggingface_hub import get_token, hf_hub_url
    url = hf_hub_url("ai4bharat/IndicCorpV2", "data/ne.txt", repo_type="dataset")
    download_range(url, "data/ref/words.raw", 6_000_000_000, WORDLIST_MB * 1_000_000, get_token())
    download_range(url, "data/ref/human.raw", 9_000_000_000, 5_000_000, get_token())
    print("word-list lines:", clean_lines("data/ref/words.raw", "data/ref/words.txt"))
    print("reference lines:", clean_lines("data/ref/human.raw", "data/ref/human.txt", max_lines=5000))
    os.remove("data/ref/words.raw"); os.remove("data/ref/human.raw")

## Step 2. Run both tests
Part 1 (generation) runs on GPU 0 and part 2 (classification) on GPU 1 at the same time. Progress is printed every 2 minutes; full logs are saved in `results/eval_logs/`.

In [ ]:
os.makedirs("results/eval_logs", exist_ok=True)
models_glob = os.path.join(MODEL_DIR, "*.pt")
JOBS = {
    "generation": f"python scripts/generate_eval.py --models '{models_glob}' --tokenizers {TOK_DIR} "
                  f"--prompts data/prompts/flores_v1_test.ne --n {N_PROMPTS} --prompt-words {PROMPT_WORDS} "
                  f"--max-new {MAX_NEW} --wordlist-text data/ref/words.txt --wordlist-mb {WORDLIST_MB} "
                  f"--reference data/ref/human.txt --out results/generation",
    "classification": f"python scripts/finetune_cls.py --models '{models_glob}' --tokenizers {TOK_DIR} "
                      f"--train-per-class {CLS_TRAIN_PER_CLASS} --test-per-class {CLS_TEST_PER_CLASS} "
                      f"--epochs {CLS_EPOCHS} --cache-dir {WORK}/hf_cache --out results/classification",
}


def tail(path, n=1):
    try:
        lines = [l for l in open(path, encoding="utf-8", errors="replace").read().splitlines() if l.strip()]
        return lines[-n:]
    except FileNotFoundError:
        return []


procs, t0 = {}, time.time()
for i, (name, cmd) in enumerate(JOBS.items()):
    env = dict(os.environ)
    if N_GPU:
        env["CUDA_VISIBLE_DEVICES"] = str(i % N_GPU)
    log = f"results/eval_logs/{name}.log"
    procs[name] = (subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT), log)
    print(f"started {name} on GPU {i % max(1, N_GPU)}", flush=True)
    if N_GPU < 2:                                       # one GPU: run one after the other
        procs[name][0].wait()
last = 0
while any(p.poll() is None for p, _ in procs.values()):
    if time.time() - last > 120:
        last = time.time()
        for name, (p, log) in procs.items():
            if p.poll() is None:
                print(f"[{(time.time() - t0) / 60:5.1f} min] {name}: {(tail(log) or [''])[-1].strip()[:150]}", flush=True)
    time.sleep(10)
for name, (p, log) in procs.items():
    print(f"\n===== {name}: {'done' if p.returncode == 0 else 'FAILED'} =====")
    print("\n".join(tail(log, 40)))

## Step 3. Results

In [ ]:
from IPython.display import Markdown, display

for title, f in [("Generated text (lower malformed / non-word = better)", "results/generation/generation_metrics.md"),
                 ("News classification (higher = better)", "results/classification/cls_metrics.md"),
                 ("Classification skipped", "results/classification/SKIPPED.txt")]:
    if os.path.exists(f):
        display(Markdown(f"### {title}\n\n" + open(f, encoding="utf-8").read()))

# a few generated samples, same opening for each model
samples = sorted(glob.glob("results/generation/*__s1.samples.txt"))
for i in range(3):
    print("-" * 80)
    for f in samples:
        lines = open(f, encoding="utf-8").read().splitlines()
        if i < len(lines):
            p, t = lines[i].split(" ||| ", 1)
            if f == samples[0]:
                print("OPENING:", p)
            print(f"  {os.path.basename(f).replace('.samples.txt', ''):35s} {t[:160]}")

## Step 4. Save the results
`aksharapiece_eval_results.zip` in the **Output** panel (small): metrics, generated samples, logs. **Send this zip back.**

In [ ]:
bundle = os.path.join(WORK, "eval_bundle")
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for d in ("generation", "classification", "eval_logs"):
    if os.path.isdir(f"results/{d}"):
        shutil.copytree(f"results/{d}", os.path.join(bundle, d))
json.dump({"settings": {k: globals()[k] for k in ("N_PROMPTS", "PROMPT_WORDS", "MAX_NEW", "WORDLIST_MB",
                                                   "CLS_TRAIN_PER_CLASS", "CLS_TEST_PER_CLASS", "CLS_EPOCHS")},
           "models": [os.path.basename(m) for m in MODELS],
           "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)]},
          open(os.path.join(bundle, "run_info.json"), "w"), indent=2)
shutil.make_archive(os.path.join(OUT, "aksharapiece_eval_results"), "zip", bundle)
shutil.rmtree(bundle)
print("Saved", os.path.join(OUT, "aksharapiece_eval_results.zip"))

## How to read the results

- **Malformed / 1000 aksharas:** syllables that cannot be written correctly. Human text scores close to 0. AksharaPiece cannot produce most of these by construction; plain BPE can, because it can cut a vowel sign off its consonant.
- **Non-words:** generated words never seen (3+ times) in 200 MB of real Nepali. Human text also has some (names, rare words), so compare with the human row.
- **Classification:** if AksharaPiece and plain BPE are within about one standard deviation, the tokenizer does not hurt (or help) understanding; that is still a useful result.